# 📗 부록: RAG 단계별 기법 체계

오늘 고른 아키텍처를 RAG의 다섯 단계와 흐름·운영 층 위에 놓고, 앞 단원 기법이 어디에 들어가는지 정리합니다. 기법 자체는 다시 설명하지 않습니다.

- **오늘의 결론:** 임베딩 모델 KURE-v1, 벡터 저장소 pgvector, 검색 전략은 질문 유형별 전략입니다.
- **읽는 법:** 1절은 세 선택의 기반이 된 앞 단원 기법, 2절은 단계마다 붙일 수 있는 기법, 3절은 기법을 붙이기 전에 다시 평가할 것입니다.

## 1. 세 선택의 기반이 된 앞 단원 기법

세 가지 선택은 새 도구가 아니라 앞 단원에서 배운 기법을 여행지 검색 서비스의 평가셋(관광지 질문 75개)으로 다시 비교해 고른 것입니다.

<img src="images/10_appendix_three_choices_map.png" width="1050" alt="세 선택에 들어간 앞 단원 기법: 임베딩 모델 KURE-v1에는 코사인 유사도 검색, 평가셋과 지표, BM25가, 벡터 저장소 pgvector에는 Chroma·Qdrant, 메타데이터 필터, SQL JOIN, Neo4j 벡터 인덱스가, 검색 전략 유형별 전략에는 하이브리드 검색, 관계 확장, SQL 집계가 꼬리표로 붙어 있습니다.">

| 선택 | 기반이 된 앞 단원 기법 | 오늘의 결론 |
|---|---|---|
| 임베딩 모델 | 코사인 유사도 검색, 평가셋과 지표, BM25 기준선 | KURE-v1 |
| 벡터 저장소 | Chroma·Qdrant, 메타데이터 필터, SQL JOIN, Neo4j 벡터 인덱스 | pgvector |
| 검색 전략 | 하이브리드 검색, 관계 확장, SQL 집계 | 질문 유형별 전략 |

## 2. 단계마다 붙일 수 있는 기법

RAG를 다섯 단계와, 단계 전체를 감싸는 **흐름·운영 층**으로 나눕니다. Modular RAG 단원에서 배운 모듈 구성을 따른 틀이며, 기법은 **어디에서 무엇을 바꾸는지**로 나눕니다.

- **색인**: 질문이 오기 전에 문서를 나누고 저장하는 방식을 바꿉니다.
- **질문 처리**: 검색 전에 질문을 바꾸거나 경로를 정합니다.
- **검색**: 후보를 찾는 방법을 바꾸거나, 근거가 부족하면 다시 찾습니다.
- **결과 다듬기**: 찾은 결과의 순위를 다시 매기고(여러 결과의 순위 합치기 포함), LLM에 넘길 범위와 길이를 고칩니다.
- **답변**: 만든 답을 검토하고 고칩니다.
- **흐름·운영(층)**: 위 단계를 잇고 분기·반복하며, 기억, 역할 분담, 관측과 실패 처리를 맡습니다.

<img src="images/11_appendix_pipeline.png" width="1050" alt="고른 아키텍처에 붙일 수 있는 기법: 색인, 질문 처리, 검색, 결과 다듬기, 답변 다섯 단계 띠 위에 오늘 고른 것(KURE-v1 + pgvector, 유형 라벨, 유형별 전략, 다듬기 없음, 선정 범위 밖)이 실선 카드로, 아래에 붙일 수 있는 기법(색인: 청킹 전략, RAPTOR, GraphRAG·LightRAG / 질문 처리: Self-Query, HyDE·Multi-Query, Step-back·질문 분해, 질문 라우팅 / 검색: 하이브리드 검색, 그래프 확장 검색, Text-to-Cypher, CRAG / 결과 다듬기: 리랭킹, RAG-Fusion, 문맥 확장, 컨텍스트 압축 / 답변: Self-RAG 스타일)이 측정 후 추가를 뜻하는 점선 상자로 놓여 있고, 맨 아래 흐름·운영 띠에 LangGraph, 기억, 멀티에이전트, 관측·실패 처리가 있습니다. 질문 처리의 Self-Query와 질문 라우팅은 실제 운영에 꼭 필요한 부품입니다.">

### 2-1. 단계마다 오늘 고른 것

| 단계 | 하는 일 | 오늘 고른 것 |
|---|---|---|
| 색인 | 문서를 나누고 저장 | 문서 1건 = 벡터 1개(KURE-v1, 1024차원 float32)를 pgvector에, 이름 질문용 BM25 색인 |
| 질문 처리 | 검색 전에 질문을 바꾸거나 경로를 정함 | 평가셋의 유형 라벨·조건·기준 관광지를 그대로 씀 |
| 검색 | 후보를 찾음 | 유형별 전략(의미 검색, BM25, 지역 필터 후 검색, 근처 관계 확장, 표 집계) |
| 결과 다듬기 | 찾은 결과를 LLM에 넘기기 좋게 고침 | 다듬기 없음(검색 순위 그대로) |
| 답변 | 만든 답을 검토하고 고침 | 선정 범위 밖(검색 단계까지만 평가해 고름) |
| 흐름·운영 | 단계를 잇고 운영 | 선정 범위 밖 |

실제 질문에는 유형 라벨·조건·기준 관광지가 적혀 있지 않습니다. 그래서 질문 처리의 **경로 선택과 조건 추출은 측정 후 추가가 아니라 운영에 꼭 필요한 부품**입니다.

### 2-2. 앞 단원에서 배운 기법 한눈에 보기

그림에는 대표 기법만 넣었고, 아래 표가 전체입니다.

| 단계 | 범주 | 기법 | 언제 붙이나 |
|---|---|---|---|
| **색인** | 청킹 | Fixed, Recursive, Semantic | 문서가 최대 입력에서 잘리거나, 질문이 묻는 범위와 청크 크기가 맞지 않을 때 |
| | 요약 계층 | RAPTOR | 여러 절에 흩어진 내용을 종합하는 질문이 세부 질문과 함께 들어올 때 |
| | 그래프 색인 | Microsoft GraphRAG, LazyGraphRAG | 자료 전체를 모아 답하는 전역 질문이 많을 때 |
| | | LightRAG, HippoRAG | 관계 표 없이 문서를 이어 읽는 멀티홉 질문이 많을 때 |
| **질문 처리** | 조건 추출 | Self-Query | 질문 속 지역 같은 조건을 필터로 바꿀 때(운영 필수) |
| | 질의 변환 | HyDE, Multi-Query, Step-back, 질문 분해 | 표현이 문서와 달라 놓칠 때, 배경 원리가 필요하거나 확인할 항목이 여럿일 때 |
| | 경로 선택 | 질문 분류 라우팅, Jev 분기, Adaptive RAG | 실제 질문에 유형 라벨이 없을 때(운영 필수) |
| **검색** | 하이브리드 검색 | BM25 + 의미 검색(RRF, EnsembleRetriever) | 낱말 일치와 뜻 일치가 둘 다 필요할 때 |
| | 그래프 확장 검색 | 벡터 진입 + 관계 확장(VectorCypherRetriever), 전문·벡터·관계 통합 | 관계를 따라가야 답이 모일 때 |
| | 구조화 질의 | Text-to-Cypher(GraphCypherQAChain), 표 집계 | 세거나 조건으로 정확히 골라야 할 때 |
| | 검색 교정 | CRAG | 검색 근거가 부족한 질문이 잦을 때(근거 평가 후 질문 재작성·재검색·웹 검색) |
| **결과 다듬기** | 재정렬 | 리랭킹(Cross-Encoder, 재순위로 쓰는 ColBERT), PageRank 재정렬 | 정답이 후보 안에 있는데 5위 밖일 때, 그래프 중심성을 순위에 반영할 때 |
| | 순위 융합 | RAG-Fusion | 질문 하나로는 근거가 일부만 잡힐 때 |
| | 문맥 확장 | Sentence Window, Parent-Child, Auto-merging | 찾은 조각이 짧아 앞뒤 문맥이나 상위 단락이 필요할 때 |
| | 압축 | 컨텍스트 압축(LLM 추출) | 넘길 문맥이 길 때 |
| **답변** | 답변 검토·수정 | Self-RAG 스타일 | 답이 근거를 벗어나거나 질문에 덜 답하기 쉬울 때 |
| **흐름·운영** | 그래프 흐름 | LangGraph(조건 분기, 사이클, 체크포인터, HITL, 스트리밍) | 경로가 여럿이거나 검토·재시도·사람 승인이 필요할 때 |
| | 기억 | Store, Mem0, 컨텍스트 엔지니어링 | 대화가 이어지고 사용자 기억이나 입력 분량 조절이 필요할 때 |
| | 멀티에이전트 | 슈퍼바이저·워커, 동적 계획, 병렬 조사, 핸드오프 | 일이 여러 전문 작업으로 나뉠 때 |
| | 관측·실패 처리 | Langfuse 관측·궤적 디버깅, 핑퐁·재시도·타임아웃, 부분 실패·폴백·이어 하기 | 운영 중 실패를 찾고 견뎌야 할 때 |

**표를 읽을 때**
- 하이브리드 검색과 RAG-Fusion은 둘 다 RRF를 씁니다. 하이브리드 검색은 BM25와 의미 검색을 한 검색기로 묶은 것이라 검색에, RAG-Fusion은 질의 변환으로 만든 여러 질의의 결과를 RRF로 다시 순위 매기는 것이라 결과 다듬기에 둡니다. RAG-Fusion 단원은 이 융합을 검색 모듈 안에서 구현했습니다.
- 문맥 확장은 준비를 색인 때(작은 조각과 함께 부모나 앞뒤 문장을 저장) 하고, 효과는 검색 뒤에 납니다.
- 오늘 이미 쓴 것: 하이브리드 검색은 후보 전략으로 평가했고(이름 질문은 BM25만으로 충분), 그래프 확장 검색은 근처 관계 표로 한 단계만(`expand_near`), 구조화 질의는 표 집계(`count_with_pandas`)로 썼습니다.
- 모두 측정한 뒤에 붙입니다. 다만 경로 선택과 조건 추출은 실제 질문을 받으려면 꼭 필요합니다.

## 3. 붙이기 전에 다시 평가할 것

앞 단원 기법은 그 단원의 데이터와 시스템에서 효과를 확인한 것입니다. 우리 도메인의 문서·질문과 우리 시스템의 제약(지연·비용·저장 공간)에서도 더 나은지는 교안_02 6절의 선정 틀(제약으로 거르기, 지배당한 후보 지우기, 고정 기준 가중 점수, 민감도 확인)로 다시 평가해야 알 수 있습니다.

<img src="images/12_appendix_reselect_loop.png" width="1050" alt="기법을 붙이면 같은 틀로 다시 고르기: 새 후보 추가, 같은 평가셋으로 평가, 네 축 비교, 제약·지배·가중 점수, 민감도 확인, ADR 갱신 여섯 단계가 고리로 이어집니다.">

**같은 틀에 새 후보로 넣기**
- 기법을 붙인 구성은 후보표의 새 행입니다.
- 같은 평가셋·네 축·제약·고정 기준과 가중치로 점수를 매기고 민감도까지 봅니다. 1위가 바뀌면 ADR을 새로 남깁니다.

**기법마다 더해지는 부담**

| 부담 | 해당 기법 | 확인할 것 |
|---|---|---|
| 질문마다 LLM 호출 | Self-Query, 질의 변환, RAG-Fusion, LLM 라우터·Jev, Text-to-Cypher, 컨텍스트 압축, CRAG, Self-RAG 스타일, Mem0 기억 추출, 멀티에이전트, 그래프 검색 일부(LightRAG, HippoRAG, LazyGraphRAG, GraphRAG Global·DRIFT) | 검색 단계 300ms와 월 비용 기준 안인지. 외부 LLM·Jev·웹 검색 같은 외부 API는 외부 API 금지 제약에서 먼저 걸립니다 |
| 질문마다 계산 추가 | 리랭킹(Cross-Encoder, ColBERT) | 지연 |
| LLM에 넘기는 토큰 증가 | 문맥 확장 | 입력 토큰 수와 비용 |
| 색인을 다시 만듦 | 청킹 변경, 문맥 확장 준비, RAPTOR, 그래프 색인, 임베딩 모델 교체 | 벡터 수·저장 공간, 색인 때 LLM 비용(LazyGraphRAG는 0회), 유형 × 전략 정확도 표 다시 만들기 |

**평가셋 정답 대신 라우터와 Self-Query를 쓸 때**
- 교안_02는 검색 전에 정해야 하는 두 값을 질문에서 뽑지 않고 평가셋에 적힌 정답을 그대로 썼습니다. 질문 유형(`question_type`)과 조건 질문의 지역 조건(`filter`)입니다.
- 그래서 유형별 정확도(0.899)는 두 값이 하나도 틀리지 않을 때의 상한입니다.
- 실제 서비스에서는 두 값을 질문에서 직접 얻어야 하고, 기법이 하나씩 맡습니다.

| 평가셋에서 가져온 값 | 대신 맡는 기법 | 따로 측정할 정확도 |
|---|---|---|
| 질문 유형(`question_type`) | 라우터: 질문을 다섯 유형 가운데 하나로 분류해 검색 전략을 고름 | 분류 정확도: 라우터가 고른 유형이 평가셋 유형과 같은 질문의 비율 |
| 지역 조건(`filter`) | Self-Query: 조건 질문에서 지역 같은 메타데이터 필터를 뽑음 | 조건 추출 정확도: 뽑은 필터가 평가셋 `filter`와 같은 질문의 비율 |

- 라우터가 유형을 틀리면 엉뚱한 전략으로, Self-Query가 조건을 틀리면 엉뚱한 지역으로 검색합니다. 그래서 두 정확도를 따로 측정한 뒤, 평가셋 정답 대신 두 기법의 결과로 검색을 다시 돌려 후보표의 정확도를 다시 계산합니다.